# WC2026 Predictor — Sequence Model for International Football Outcomes

A deep learning project predicting FIFA World Cup match outcomes, built around four testable hypotheses about how international football has changed since the 2006-2010 era.

**Critical methodology note:** Training data is restricted to 1990–2022. The 2026 tournament is used purely as a held-out, real-time evaluation set — pre-tournament predictions are locked in BEFORE comparing against actual results, exactly mirroring walk-forward validation used in quantitative finance research.

See `README.md` for full hypothesis-to-feature mapping and architecture rationale.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/wc2026')
import sys
sys.path.insert(0, '/content/drive/MyDrive/wc2026/src')

Mounted at /content/drive


In [ ]:
import os
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_5ccaa88e724da0ebaea353cf52c8be3c'
!pip install kagglehub torch scikit-learn matplotlib pandas numpy --quiet
import kagglehub

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## 1. Data Download & Loading

First time running this in Colab, `kagglehub` will prompt for Kaggle credentials. Easiest path: go to kaggle.com → Account → Create New API Token, upload the resulting `kaggle.json` when prompted, or use `kagglehub.login()`.

In [ ]:
from data_loader import (
    download_datasets, load_match_history, load_elo_ratings,
    load_fifa_rankings, load_wc2026_live, build_team_match_long_format,
    is_host_nation_home_match, WC2026_HOST_NATIONS
)

paths = download_datasets()

Using Colab cache for faster access to the 'international-football-results-from-1872-to-2017' dataset.


100%|██████████| 127k/127k [00:00<00:00, 62.0MB/s]

Extracting files...


100%|██████████| 1.69M/1.69M [00:00<00:00, 109MB/s]

Extracting files...


Using Colab cache for faster access to the 'fifa-world-cup-2026-dataset' dataset.


In [ ]:
matches_df = load_match_history(paths['matches'])
elo_df = load_elo_ratings(paths['elo'])
fifa_rank_df = load_fifa_rankings(paths['fifa_rank'])
wc2026_live_df = load_wc2026_live(paths['wc2026'])

print(matches_df[['date','home_team','away_team','home_score','away_score','tournament','neutral']].head())
print(elo_df[['year','team','rating']].head())
print(fifa_rank_df[['team','rank','rank_date']].head())
print(wc2026_live_df[['home_team','away_team','home_score','away_score','status','stage_name']].head())

        date home_team away_team  home_score  away_score tournament  neutral
0 1872-11-30  Scotland   England         0.0         0.0   Friendly    False
1 1873-03-08   England  Scotland         4.0         2.0   Friendly    False
2 1874-03-07  Scotland   England         2.0         1.0   Friendly    False
3 1875-03-06   England  Scotland         2.0         2.0   Friendly    False
4 1876-03-04  Scotland   England         3.0         0.0   Friendly    False
   year       team  rating
0  1901    England    2013
1  1901   Scotland    1973
2  1902  Argentina    2021
3  1902    England    1995
4  1902   Scotland    1983
                team   rank  rank_date
0  Brunei Darussalam  140.0 1992-12-31
1           Portugal   33.0 1992-12-31
2             Zambia   32.0 1992-12-31
3             Greece   31.0 1992-12-31
4            Algeria   30.0 1992-12-31
     home_team               away_team  home_score  away_score     status  \
0       Mexico            South Africa         2.0         0.0  C

In [ ]:
long_matches_df = build_team_match_long_format(matches_df)
print(f"Long format: {len(long_matches_df):,} rows")

Long format: 98,954 rows


## 2. Exploratory Analysis — Validate Hypotheses BEFORE Building The Model

Before building anything, check whether the hypotheses actually hold in the data. If "defending champions doing better since 2018" isn't real, don't build a feature pretending it is.

In [ ]:
PAST_WINNERS = {
    1990: 'Germany', 1994: 'Brazil', 1998: 'France', 2002: 'Brazil',
    2006: 'Italy', 2010: 'Spain', 2014: 'Germany', 2018: 'France', 2022: 'Argentina'
}

wc_matches = matches_df[matches_df['tournament'].str.contains('World Cup', case=False, na=False)]
wc_matches = wc_matches[~wc_matches['tournament'].str.contains('qualif', case=False, na=False)]
print(f"World Cup matches found: {len(wc_matches)}")

World Cup matches found: 1096


## 3. Feature Engineering — Build The Training Dataset

In [ ]:
from features import (
    assemble_match_features, CONTINENTAL_CHAMPIONS,
    compute_recent_continental_title_feature
)

In [ ]:
def build_features_for_wc_matches(match_subset, matches_long_data):
    rows = []
    skipped = 0
    for _, match in match_subset.iterrows():
        try:
            is_neutral = bool(match['neutral']) if not isinstance(match['neutral'], str) else match['neutral'] == 'True'
            feats = assemble_match_features(
                home_team=match['home_team'],
                away_team=match['away_team'],
                match_date=match['date'],
                matches_long=matches_long_data[matches_long_data['date'] < match['date']],
                elo_df=elo_df,
                fifa_rank_df=fifa_rank_df,
                past_winners=PAST_WINNERS,
                is_neutral_venue=is_neutral,
            )
            feats['outcome'] = match['outcome']
            feats['year'] = match['year']
            rows.append(feats)
        except Exception as e:
            skipped += 1
    print(f"  Built {len(rows)} rows, skipped {skipped}")
    return pd.DataFrame(rows)

wc_train_matches = wc_matches[wc_matches['date'] < pd.Timestamp('2023-01-01')].copy()
features_df = build_features_for_wc_matches(wc_train_matches, long_matches_df)
print(f"features_df shape: {features_df.shape}")
print(features_df['is_neutral_venue'].value_counts())

  Built 1024 rows, skipped 0
features_df shape: (1024, 23)
is_neutral_venue
1    885
0    139
Name: count, dtype: int64


In [ ]:
is_friendly = matches_df['tournament'].str.contains('Friendly', case=False, na=False)
competitive_matches = matches_df[~is_friendly].copy()

def build_features_for_matches(match_subset, matches_long_data):
    rows = []
    skipped = 0
    for _, match in match_subset.iterrows():
        try:
            is_neutral = bool(match['neutral']) if not isinstance(match['neutral'], str) else match['neutral'] == 'True'
            feats = assemble_match_features(
                home_team=match['home_team'],
                away_team=match['away_team'],
                match_date=match['date'],
                matches_long=matches_long_data[matches_long_data['date'] < match['date']],
                elo_df=elo_df,
                fifa_rank_df=fifa_rank_df,
                past_winners=PAST_WINNERS,
                is_neutral_venue=is_neutral,
            )
            feats['outcome'] = match['outcome']
            feats['date'] = match['date']
            rows.append(feats)
        except Exception as e:
            skipped += 1
    print(f"  Built {len(rows)} rows, skipped {skipped}")
    return pd.DataFrame(rows)

train_pool = competitive_matches[competitive_matches['date'] < pd.Timestamp('2023-01-01')].copy()
print(f"Building features for {len(train_pool)} matches (several minutes)...")
all_features_df = build_features_for_matches(train_pool, long_matches_df)
print(f"all_features_df shape: {all_features_df.shape}")
print(all_features_df['is_neutral_venue'].value_counts())

Building features for 28343 matches (several minutes)...
  Built 28343 rows, skipped 0
all_features_df shape: (28343, 23)
is_neutral_venue
0    18854
1     9489
Name: count, dtype: int64


In [ ]:
all_features_df.to_pickle('/content/drive/MyDrive/wc2026/all_features_df.pkl')
features_df.to_pickle('/content/drive/MyDrive/wc2026/features_df.pkl')
print("Saved feature datasets to Drive.")

Saved feature datasets to Drive.


In [ ]:
from model import WC2026Predictor

OUTCOME_MAP = {'home_win': 0, 'draw': 1, 'away_win': 2}

def features_to_tensors(df_subset):
    home_form = torch.tensor(np.stack(df_subset['home_form_sequence'].values), dtype=torch.long)
    away_form = torch.tensor(np.stack(df_subset['away_form_sequence'].values), dtype=torch.long)
    home_elo = torch.tensor(np.stack(df_subset['home_elo_trajectory'].values), dtype=torch.float32)
    away_elo = torch.tensor(np.stack(df_subset['away_elo_trajectory'].values), dtype=torch.float32)
    static_cols = [
        'elo_diff', 'is_neutral_venue',
        'home_is_defending_champion', 'home_defending_champion_x_modern_era',
        'away_is_defending_champion', 'away_defending_champion_x_modern_era',
        'home_is_modern_era',
        'home_won_continental_title_recently', 'away_won_continental_title_recently',
    ]
    static = torch.tensor(df_subset[static_cols].values.astype(np.float32))
    y = torch.tensor(df_subset['outcome'].map(OUTCOME_MAP).values, dtype=torch.long)
    return home_form, away_form, home_elo, away_elo, static, y

def train_model(model, train_data, n_epochs=100, lr=1e-3):
    home_form, away_form, home_elo, away_elo, static, y = train_data
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    criterion = nn.CrossEntropyLoss()
    model.train()
    for epoch in range(n_epochs):
        optimizer.zero_grad()
        logits = model(home_form, away_form, home_elo, away_elo, static)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        scheduler.step(loss.item())
        if (epoch + 1) % 20 == 0:
            print(f"  Epoch {epoch+1}/{n_epochs}, loss={loss.item():.4f}")
    return model

## 4. Walk-Forward Validation

Train on everything before WC2014 → test on WC2014. Then include WC2014, test on WC2018. Then include WC2018, test on WC2022.

In [ ]:
from evaluate import evaluate_predictions, walk_forward_summary

WC_START_DATES = {
    2014: pd.Timestamp('2014-06-12'),
    2018: pd.Timestamp('2018-06-14'),
    2022: pd.Timestamp('2022-11-20'),
}

results_by_tournament = {}
for test_year, cutoff in WC_START_DATES.items():
    print(f"\n--- Walk-forward fold: test on WC{test_year} ---")
    train_subset = all_features_df[all_features_df['date'] < cutoff].reset_index(drop=True)
    test_subset = features_df[features_df['year'] == test_year].reset_index(drop=True)
    print(f"  Training: {len(train_subset):,}  |  Test: {len(test_subset)}")
    train_data = features_to_tensors(train_subset)
    test_data = features_to_tensors(test_subset)
    torch.manual_seed(42)
    model = WC2026Predictor(static_feature_dim=9)
    model = train_model(model, train_data, n_epochs=100)
    model.eval()
    with torch.no_grad():
        home_form, away_form, home_elo, away_elo, static, y_true = test_data
        logits = model(home_form, away_form, home_elo, away_elo, static)
        probs = torch.softmax(logits, dim=1).numpy()
    results = evaluate_predictions(y_true.numpy(), probs)
    results_by_tournament[test_year] = results
    print(f"  log_loss={results['log_loss']:.4f}, accuracy={results['accuracy']:.3f}")

walk_forward_summary(results_by_tournament)


--- Walk-forward fold: test on WC2014 ---
  Training: 22,818  |  Test: 64
  Epoch 20/100, loss=1.2518
  Epoch 40/100, loss=1.1165
  Epoch 60/100, loss=1.0583
  Epoch 80/100, loss=1.0221
  Epoch 100/100, loss=1.0086
  log_loss=1.0045, accuracy=0.594

--- Walk-forward fold: test on WC2018 ---
  Training: 25,277  |  Test: 64
  Epoch 20/100, loss=1.2332
  Epoch 40/100, loss=1.1199
  Epoch 60/100, loss=1.0583
  Epoch 80/100, loss=1.0260
  Epoch 100/100, loss=1.0068
  log_loss=1.0122, accuracy=0.562

--- Walk-forward fold: test on WC2022 ---
  Training: 28,263  |  Test: 64
  Epoch 20/100, loss=1.2426
  Epoch 40/100, loss=1.1110
  Epoch 60/100, loss=1.0609
  Epoch 80/100, loss=1.0194
  Epoch 100/100, loss=1.0102
  log_loss=0.9837, accuracy=0.531

Tournament  N Matches   Log Loss    Mean Brier  Accuracy  
----------------------------------------------------------------------
2014        64          1.0045      0.1988      0.594     
2018        64          1.0122      0.2032      0.562     
2

## 5. Final Model — Train on All Data Through 2022

In [ ]:
final_train_data = features_to_tensors(all_features_df)
torch.manual_seed(42)
final_model = WC2026Predictor(static_feature_dim=9)
final_model = train_model(final_model, final_train_data, n_epochs=100)
print(f"Final model trained on {len(all_features_df):,} matches.")

  Epoch 20/100, loss=1.2339
  Epoch 40/100, loss=1.1115
  Epoch 60/100, loss=1.0565
  Epoch 80/100, loss=1.0245
  Epoch 100/100, loss=1.0102
Final model trained on 28,343 matches.


In [ ]:
torch.save(final_model.state_dict(), '/content/drive/MyDrive/wc2026/final_model_weights.pt')
print("Saved model weights to Drive.")

Saved model weights to Drive.


## 6. Monte Carlo Tournament Simulations

Generate predictions using Monte Carlo Simulations for 2026 matches using ONLY pre-tournament data (squad strength, qualifying form, Elo entering the tournament) — before looking at any 2026 results.

In [ ]:
teams_csv = list(Path(paths['wc2026']).glob('teams.csv'))[0]
teams_df = pd.read_csv(teams_csv)

groups = {}
for group_letter, group_df in teams_df.groupby('group_letter'):
    groups[group_letter] = group_df['team_name'].tolist()

fifa_ranks = dict(zip(teams_df['team_name'], teams_df['fifa_ranking_pre_tournament']))
print(f"Built {len(groups)} groups, {sum(len(v) for v in groups.values())} teams")

Built 12 groups, 48 teams


In [ ]:
from bracket import run_monte_carlo_simulation

WC2026_START = pd.Timestamp('2026-06-11')
pre_tournament_long = long_matches_df[long_matches_df['date'] < WC2026_START]

results = run_monte_carlo_simulation(
    groups=groups,
    model=final_model,
    matches_long=pre_tournament_long,
    elo_df=elo_df,
    fifa_rank_df=fifa_rank_df,
    past_winners=PAST_WINNERS,
    fifa_ranks=fifa_ranks,
    is_neutral_venue_fn=is_host_nation_home_match,
    match_date=WC2026_START,
    assemble_match_features_fn=assemble_match_features,
    n_simulations=1000,
    progress_every=100,
)

print(results[['team','pct_champion','pct_final','pct_semi_finals','pct_quarter_finals']].head(15))

  Completed 100/1000 simulations
  Completed 200/1000 simulations
  Completed 300/1000 simulations
  Completed 400/1000 simulations
  Completed 500/1000 simulations
  Completed 600/1000 simulations
  Completed 700/1000 simulations
  Completed 800/1000 simulations
  Completed 900/1000 simulations
  Completed 1000/1000 simulations
           team  pct_champion  pct_final  pct_semi_finals  pct_quarter_finals
0         Spain         0.129      0.184            0.302               0.516
1        France         0.098      0.178            0.294               0.417
2     Argentina         0.098      0.183            0.257               0.358
3        Brazil         0.078      0.147            0.229               0.475
4   Netherlands         0.060      0.087            0.206               0.339
5       Germany         0.053      0.123            0.272               0.423
6       Ecuador         0.045      0.112            0.209               0.360
7       England         0.043      0.082     

In [ ]:
results.to_csv('/content/drive/MyDrive/wc2026/monte_carlo_results.csv', index=False)
print("Saved Monte Carlo results to Drive.")

Saved Monte Carlo results to Drive.


## 7. Live Scoring — Compare Predictions Against Actual 2026 Results

Since the tournament is ~40% complete, score the pre-tournament predictions against real outcomes that have already happened.

In [ ]:
completed = predictions_df.dropna(subset=['actual_home_score', 'actual_away_score']).copy()
completed['actual_outcome'] = np.select(
    [completed['actual_home_score'] > completed['actual_away_score'],
     completed['actual_home_score'] < completed['actual_away_score']],
    ['p_home_win', 'p_away_win'],
    default='p_draw'
)

print(f"Completed matches so far: {len(completed)}")

y_true_2026 = completed['actual_outcome'].map({'p_home_win':0, 'p_draw':1, 'p_away_win':2}).values
y_pred_proba_2026 = completed[['p_home_win','p_draw','p_away_win']].values

results_2026 = evaluate_predictions(y_true_2026, y_pred_proba_2026)
print(f"\nWC2026 live scoring (pre-tournament predictions vs actual results so far):")
print(f"  Log loss: {results_2026['log_loss']:.4f}")
print(f"  Mean Brier: {results_2026['mean_brier']:.4f}")
print(f"  Accuracy: {results_2026['accuracy']:.3f}")

baseline_2026 = baseline_constant_prediction(y_true_2026)
print(f"\nNaive baseline for comparison: log_loss={baseline_2026['log_loss']:.4f}")

Generated and saved 72 predictions


In [ ]:
# Run this in a cell to force a manual checkpoint, in addition to File > Save
from IPython.display import Javascript
display(Javascript('google.colab.kernel.invokeFunction("checkpoint", [], {})'))

<IPython.core.display.Javascript object>